In [1]:
%load_ext autoreload
%autoreload 2
%load_ext jupyter_black

In [2]:
import os, sys

os.chdir("..")
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
sys.path.insert(0, ".")

In [14]:
from tests.test_lstar import *
from orthogonal_dfa.l_star.cluster import sample_suffix_family

In [143]:
oracle_creator = AllFramesClosedOracle
min_signal_strength = 0.4
seed = 0
noise_model = None
min_suffix_frequency = 0.05
pst = compute_pst(
    oracle_creator,
    min_signal_strength,
    seed,
    noise_model=noise_model,
    min_suffix_frequency=min_suffix_frequency,
)

Using suffix population size 29, eps 0.22666666666666668, and 780 prefixes (signal strength 0.4).


In [149]:
v_idx = pst.table.intern_suffix([])
vs, boundary = sample_suffix_family(pst, v_idx, first_round=True)

FNR 0.0324 too high, sampling more suffixes; decision_boundary: 0.4863
FNR 0.0246 too high, sampling more suffixes; decision_boundary: 0.4897
FNR 0.0257 too high, sampling more suffixes; decision_boundary: 0.4977
FNR 0.0238 too high, sampling more suffixes; decision_boundary: 0.4976
FNR 0.0201 too high, sampling more suffixes; decision_boundary: 0.4987
FNR limit reached, decision boundary: 0.4989, margin: 0.2267


In [289]:
# states = [set(), set()]
transitions = [{}, {}]
transition_witnesess = {}
# accepts = [None, None]
dt = ((), {True: 0, False: 1})

In [290]:
def is_accept(seq, prepend_to_suffixes):
    seq = list(seq)
    prepended_vs = [
        pst.table.intern_suffix([*prepend_to_suffixes, *pst.table.suffix(v)])
        for v in vs
    ]
    pst.table.ensure_prefixes([seq])
    [[rej], [acc]] = pst.compute_decision_array(
        prepended_vs, pst.table.prefix_one_hot_mask(seq)
    )
    if rej or acc:
        return acc
    print(seq)
    return None


def sift(seq, table):
    if isinstance(table, int):
        return table
    prepend, lookup = table
    decision = is_accept(seq, prepend)
    if decision is None:
        return None
    return sift(seq, lookup[decision])


def disagreement(s, sprime, table, prepend_to_tree):
    assert not isinstance(table, int), "got to the end, no disagreement"
    prepend, lookup = table
    prepend = (*prepend_to_tree, *prepend)
    decision = is_accept(s, prepend)
    decisionprime = is_accept(sprime, prepend)
    assert (
        decision is not None and decisionprime is not None
    ), "these should have been classified"
    if decision != decisionprime:
        return prepend
    return disagreement(s, sprime, lookup[decision], prepend_to_tree)


def map_dt(k, v, dt):
    if isinstance(dt, int):
        if k == dt:
            return v
        return dt
    table = dt[1]
    for decision, result in table.items():
        table[decision] = map_dt(k, v, result)
    return dt


def split(state, distinguisher):
    global transitions, transition_witnessess
    new_state = len(transitions)
    transitions = [{} for _ in range(new_state + 1)]
    transition_witnessess = {}
    # transitions[state] = {}
    # for k, v in enumerate(transitions):
    #     transitions[k] = {c: s2 for c, s2 in v.items() if s2 != state}
    # transitions.append({})
    # to_delete = [(s1, c, s2) for s1, c, s2 in transition_witnesess if state in (s1, s2)]
    # for k in to_delete:
    #     del transition_witnesess[k]
    new_dt = map_dt(state, (distinguisher, {True: state, False: new_state}), dt)
    assert new_dt is dt

In [291]:
# TODO handle indecisives better
def compute_first_disagreement(s, states, agree_point, disagree_point):
    assert 0 <= agree_point < disagree_point <= len(s), (agree_point, disagree_point)
    if agree_point + 1 == disagree_point:
        return disagree_point
    test_point = (agree_point + disagree_point) // 2
    actual_at_test = sift(tuple(s[:test_point]), dt)
    if actual_at_test is None or states[test_point] is None:
        return None
    if actual_at_test == states[test_point]:
        return compute_first_disagreement(s, states, test_point, disagree_point)
    return compute_first_disagreement(s, states, agree_point, test_point)

In [292]:
for _ in range(10000):
    to_process = us.sample(pst.rng, pst.alphabet_size)
    state = None
    agree_point = None
    states = []
    for i in range(len(to_process)):
        if state is None:
            state = sift(tuple(to_process[:i]), dt)
        states.append(state)
        if state is None:
            continue
        if agree_point is None:
            agree_point = i
        symbol = to_process[i]
        if symbol in transitions[state]:
            state = transitions[state][symbol]
            continue
        next_state = sift(tuple(to_process[: i + 1]), dt)
        if next_state is not None:
            transitions[state][symbol] = next_state
            transition_witnesess[state, symbol, next_state] = to_process[:i]
        state = next_state
    states.append(state)
    actual_state = sift(to_process, dt)
    if actual_state is None or state is None or actual_state == state:
        continue
    first_disagreement = compute_first_disagreement(to_process, states, agree_point, len(to_process))
    if first_disagreement is None:
        continue
    s1, c, s2 = (
        states[first_disagreement - 1],
        to_process[first_disagreement-1],
        states[first_disagreement],
    )
    assert s1 is not None and s2 is not None
    assert c in transitions[s1] and s2 == transitions[s1][c]
    s = transition_witnesess[s1, c, s2]
    sprime = to_process[:first_disagreement - 1]
    assert sift(s, dt) == s1
    assert sift(sprime, dt) == s1
    assert sift(s + [c], dt) == s2
    assert sift(sprime + [c], dt) != s2
    split(s1, disagreement(s, sprime, dt, [c]))
    print(transitions)

[2, 1, 0, 0, 3, 0, 2, 0, 0, 2, 1, 1, 0, 0, 3, 1, 2, 0, 1, 2, 1, 0, 0, 3, 0, 2, 0, 2, 1, 0]
[{}, {}, {}]
[0, 2, 2, 2, 2, 3, 2, 3, 3, 0, 1, 1, 3, 2, 3, 2, 0, 2, 1, 3, 1, 2, 2, 2, 3, 0, 0, 2, 2, 1, 3, 3, 0]
[0, 3, 1, 2, 3, 3, 2, 0, 0, 3, 1, 1, 0, 3, 2, 0, 3, 3, 3, 0]
[{}, {}, {}, {}]
[1, 3, 3, 3, 2, 2, 0, 0, 0, 3, 0, 2, 3, 3, 3, 2, 3, 3, 0, 2, 3, 2, 2, 3, 1, 3, 0, 3, 2, 3, 3, 3]
[{}, {}, {}, {}, {}]
[0, 3, 1, 3, 1, 3, 0, 1, 0, 1, 2, 2, 3, 2, 1, 2, 0, 2, 1, 2, 0, 3, 1, 1, 1, 3, 3, 1, 3, 3, 2, 1, 1, 3, 3, 3, 3, 3, 3, 2]
[1, 2, 3, 0, 2, 2, 3, 2, 1, 1, 0, 1, 2, 3, 3, 2, 1, 1, 1, 0, 1, 1, 0, 0, 0, 2, 1, 0, 3, 0, 0, 1, 3, 0, 3, 2, 1, 1, 1, 3]
[3, 2, 1, 0, 0, 2, 1, 0, 1, 2, 3, 2, 1, 3, 0, 2, 0, 0, 1, 0, 1, 2, 1, 0, 0, 2, 3, 3, 0, 1, 3, 0, 1, 1, 2, 3, 0, 2, 0, 3]
[0, 2, 0, 3, 3, 1, 0, 0, 3, 0, 3, 1, 0, 1, 3, 1, 0, 3, 0, 1, 1, 0, 0, 2, 3, 2, 1, 2, 0, 0, 1, 2, 0, 0, 0, 2, 1, 2, 1, 0]
[0, 1, 0, 1, 0, 2, 2, 3, 0, 3, 0, 3, 0, 0, 2, 2, 2, 3, 0, 0, 3, 0, 1, 0, 0, 3, 3, 1, 3, 0, 0, 1, 3, 3]
[1, 1, 2, 2, 

AssertionError: 

In [ ]:
transitions[5]

In [ ]:
transition_witnesess

In [264]:
states

[11,
 11,
 11,
 11,
 11,
 11,
 11,
 11,
 11,
 10,
 9,
 8,
 10,
 9,
 8,
 6,
 5,
 7,
 4,
 3,
 5,
 7,
 3,
 5,
 7,
 4,
 5,
 7,
 10,
 9,
 8,
 10,
 9,
 8,
 10,
 9,
 8,
 10,
 9,
 8,
 10]

In [239]:
transition_witnesess

{(1, 0, 1): [], (1, 1, 1): [0], (1, 2, 1): [0, 1, 1], (1, 3, 1): [0, 1, 1, 2]}

In [247]:
# s2prime = sift(sprime + [c], dt)


In [243]:
sift(s, dt), sift(sprime, dt)

(1, 0)

In [181]:
s1, c, s2, s2prime

(1, 1, 1, 0)

In [136]:
first_disagreement()

In [118]:
actual_state

0

In [89]:
symbol

2

In [123]:
agree_point

0

In [93]:
sift(tuple(to_process[:i]), dt)

1

[144,
 152,
 77,
 109,
 113,
 189,
 35,
 199,
 58,
 102,
 22,
 115,
 161,
 127,
 192,
 111,
 4,
 92,
 177,
 0,
 52,
 100,
 190,
 150,
 220,
 28,
 98,
 82,
 76]